In [ ]:
import faiss
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
INDEX_FILE = 'book_vectors.index'
DATA_FILE = 'books_data.csv'
MODEL_NAME = 'all-MiniLM-L6-v2'

def main():
    print("--- 📚 AI Book Search Engine ---")
    
    # 1. Load the Database (FAISS Index)
    try:
        index = faiss.read_index(INDEX_FILE)
        print(f"✅ Loaded Index ({index.ntotal} books)")
    except Exception as e:
        print(f"❌ Error: Could not load {INDEX_FILE}. Make sure it's in this folder!")
        return

    # 2. Load the Data (To get Titles from IDs)
    try:
        df = pd.read_csv(DATA_FILE)
        print(f"✅ Loaded Metadata ({len(df)} records)")
    except Exception as e:
        print(f"❌ Error: Could not load {DATA_FILE}")
        return

    # 3. Load the Brain (To convert queries to vectors)
    print("⏳ Loading AI Model... (One-time setup)")
    model = SentenceTransformer(MODEL_NAME)
    print("✅ Model Ready!")

    # 4. SEARCH LOOP
    print("\n👉 Type a query (e.g., 'wizards school' or 'love story in paris'). Type 'exit' to quit.\n")
    
    while True:
        user_query = input("🔍 Your Query: ")
        
        if user_query.lower() in ['exit', 'quit', 'q']:
            break
            
        # A. Convert query to vector
        query_vector = model.encode([user_query]) # Shape (1, 384)
        
        # B. Search FAISS (Find 5 closest neighbors)
        k = 5
        distances, indices = index.search(np.array(query_vector).astype('float32'), k)
        
        # C. Display Results
        print(f"\n--- Top {k} Recommendations ---")
        for i in range(k):
            book_idx = indices[0][i]  # The Row ID from FAISS
            distance = distances[0][i] # How far apart they are (Lower = Better)
            
            # Get the book details from our CSV
            result_book = df.iloc[book_idx]
            
            print(f"{i+1}. {result_book['title']}")
            # print(f"   (Confidence Score: {distance:.4f})") # Optional: See the math
            print(f"   Summary Snippet: {str(result_book['plot'])[:100]}...\n")

if __name__ == "__main__":
    main()

--- 📚 AI Book Search Engine ---
✅ Loaded Index (10000 books)
✅ Loaded Metadata (10000 records)
⏳ Loading AI Model... (One-time setup)
✅ Model Ready!

👉 Type a query (e.g., 'wizards school' or 'love story in paris'). Type 'exit' to quit.



🔍 Your Query:  "A story about a rich guy throwing parties for a girl he lost"



--- Top 5 Recommendations ---
1. Smashed: Story of a Drunken Girlhood
   Summary Snippet: Smashed: Story of a Drunken Girlhood by Koren Zailckas. Tags: ...

2. Girl Walks into a Bar . . .: Comedy Calamities, Dating Disasters, and a Midlife Miracle
   Summary Snippet: Girl Walks into a Bar . . .: Comedy Calamities, Dating Disasters, and a Midlife Miracle by Rachel Dr...

3. The Snowball: Warren Buffett and the Business of Life
   Summary Snippet: The Snowball: Warren Buffett and the Business of Life by Alice Schroeder. Tags: ...

4. Moneyball: The Art of Winning an Unfair Game
   Summary Snippet: Moneyball: The Art of Winning an Unfair Game by Michael   Lewis. Tags: ...

5. Most Talkative: Stories from the Front Lines of Pop Culture
   Summary Snippet: Most Talkative: Stories from the Front Lines of Pop Culture by Andy Cohen. Tags: ...



🔍 Your Query:  "Philosophy about the meaning of life and superhumans"



--- Top 5 Recommendations ---
1. Willpower: Rediscovering the Greatest Human Strength
   Summary Snippet: Willpower: Rediscovering the Greatest Human Strength by Roy F. Baumeister, John Tierney. Tags: ...

2. The Story of Philosophy: The Lives and Opinions of the World's Greatest Philosophers
   Summary Snippet: The Story of Philosophy: The Lives and Opinions of the World's Greatest Philosophers by Will Durant....

3. Essentialism: The Disciplined Pursuit of Less
   Summary Snippet: Essentialism: The Disciplined Pursuit of Less by Greg McKeown. Tags: ...

4. Darwin's Dangerous Idea: Evolution and the Meanings of Life
   Summary Snippet: Darwin's Dangerous Idea: Evolution and the Meanings of Life by Daniel C. Dennett. Tags: ...

5. Unlimited Power : The New Science Of Personal Achievement
   Summary Snippet: Unlimited Power : The New Science Of Personal Achievement by Anthony Robbins, Kenneth H. Blanchard, ...



🔍 Your Query:  "A scary clown in a sewer"



--- Top 5 Recommendations ---
1. Scary Stories to Tell in the Dark (Scary Stories, #1)
   Summary Snippet: Scary Stories to Tell in the Dark (Scary Stories, #1) by Alvin Schwartz, Stephen Gammell. Tags: ...

2. The Scarecrow (Jack McEvoy, #2)
   Summary Snippet: The Scarecrow (Jack McEvoy, #2) by Michael Connelly. Tags: ...

3. Captain Underpants and the Attack of the Talking Toilets (Captain Underpants, #2)
   Summary Snippet: Captain Underpants and the Attack of the Talking Toilets (Captain Underpants, #2) by Dav Pilkey. Tag...

4. The Terror
   Summary Snippet: The Terror by Dan Simmons. Tags: ...

5. The Screaming Staircase (Lockwood & Co., #1)
   Summary Snippet: The Screaming Staircase (Lockwood & Co., #1) by Jonathan Stroud. Tags: ...

